In [ ]:

import os

!rm -f plantdoc-dataset.zip
!rm -rf ./dataset

os.environ['KAGGLE_USERNAME'] = "YOUR_KAGGLE_USERNAME"
os.environ['KAGGLE_KEY'] = "YOUR_KAGGLE_API_KEY"

print("[PROCESS] Downloading real-world PlantDoc Dataset via Kaggle API...")
!kaggle datasets download -d nirmalsankalana/plantdoc-dataset

print("[PROCESS] Unpacking dataset contents into working image array matrix directories...")

!unzip -q plantdoc-dataset.zip -d ./dataset

print("[SUCCESS]")

[PROCESS] Downloading real-world PlantDoc Dataset via Kaggle API...
Dataset URL: https://www.kaggle.com/datasets/nirmalsankalana/plantdoc-dataset
License(s): CC0-1.0
100% 896M/896M [00:52<00:00, 17.8MB/s]

[PROCESS] Unpacking dataset contents into working image array matrix directories...
[SUCCESS]


In [ ]:
import tensorflow as tf

In [ ]:
H = 224
W = 224
B = 32

In [ ]:
train_path = "./dataset/train"
valid_path = "./dataset/test"

In [ ]:
train_data = tf.keras.utils.image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset='training',
    seed=123,
    image_size=(H, W),
    batch_size=B,
    label_mode='categorical'
)

valid_data = tf.keras.utils.image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset='validation',
    seed=123,
    image_size=(H, W),
    batch_size=B,
    label_mode='categorical'
)

Found 2670 files belonging to 28 classes.
Using 2136 files for training.
Found 2670 files belonging to 28 classes.
Using 534 files for validation.


In [ ]:
print(train_data.class_names)
print(len(train_data.class_names))
print(valid_data.class_names)

['Apple_Scab_Leaf', 'Apple_leaf', 'Apple_rust_leaf', 'Bell_pepper_leaf', 'Bell_pepper_leaf_spot', 'Blueberry_leaf', 'Cherry_leaf', 'Corn_Gray_leaf_spot', 'Corn_leaf_blight', 'Corn_rust_leaf', 'Peach_leaf', 'Potato_leaf_early_blight', 'Potato_leaf_late_blight', 'Raspberry_leaf', 'Soyabean_leaf', 'Squash_Powdery_mildew_leaf', 'Strawberry_leaf', 'Tomato_Early_blight_leaf', 'Tomato_Septoria_leaf_spot', 'Tomato_leaf', 'Tomato_leaf_bacterial_spot', 'Tomato_leaf_late_blight', 'Tomato_leaf_mosaic_virus', 'Tomato_leaf_yellow_virus', 'Tomato_mold_leaf', 'Tomato_two_spotted_spider_mites_leaf', 'grape_leaf', 'grape_leaf_black_rot']
28
['Apple_Scab_Leaf', 'Apple_leaf', 'Apple_rust_leaf', 'Bell_pepper_leaf', 'Bell_pepper_leaf_spot', 'Blueberry_leaf', 'Cherry_leaf', 'Corn_Gray_leaf_spot', 'Corn_leaf_blight', 'Corn_rust_leaf', 'Peach_leaf', 'Potato_leaf_early_blight', 'Potato_leaf_late_blight', 'Raspberry_leaf', 'Soyabean_leaf', 'Squash_Powdery_mildew_leaf', 'Strawberry_leaf', 'Tomato_Early_blight_lea

In [ ]:
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Rescaling,BatchNormalization,RandomFlip,RandomRotation, RandomZoom, GlobalAveragePooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2

In [ ]:
#  using mobilenet
base_model = MobileNetV2(
    input_shape=(H, W, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

model = Sequential([
    RandomFlip('horizontal_and_vertical', input_shape=(H, W, 3)),
    RandomRotation(0.2),
    RandomZoom(0.2),

    Rescaling(1./127.5, offset=-1.0),

    base_model,
    GlobalAveragePooling2D(),

    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(28, activation='softmax')
])

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_flip_6 (RandomFlip)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation_5               │ (None, 224, 224, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom_5 (RandomZoom)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling_5 (Rescaling)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 28)             │         7,196 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,593,116 (9.89 MB)

 Trainable params: 335,132 (1.28 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
checkpoint = ModelCheckpoint(
    filepath='best_plantdoc_modeltrue.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

lr_reduction = ReduceLROnPlateau(
    monitor='val_loss',
    patience=2,
    factor=0.5,
    min_lr=1e-6,
    verbose=1
)

history = model.fit(
    train_data,
    validation_data=valid_data,
    epochs=12,
    callbacks=[checkpoint, lr_reduction]
)

Epoch 1/12
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 309ms/step - accuracy: 0.0691 - loss: 3.6817
Epoch 1: val_loss improved from None to 2.93876, saving model to best_plantdoc_modeltrue.keras

Epoch 1: finished saving model to best_plantdoc_modeltrue.keras
67/67 ━━━━━━━━━━━━━━━━━━━━ 35s 428ms/step - accuracy: 0.1025 - loss: 3.4416 - val_accuracy: 0.2079 - val_loss: 2.9388 - learning_rate: 1.0000e-04
Epoch 2/12
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 322ms/step - accuracy: 0.1956 - loss: 2.9911
Epoch 2: val_loss improved from 2.93876 to 2.67096, saving model to best_plantdoc_modeltrue.keras

Epoch 2: finished saving model to best_plantdoc_modeltrue.keras
67/67 ━━━━━━━━━━━━━━━━━━━━ 27s 398ms/step - accuracy: 0.2051 - loss: 2.9303 - val_accuracy: 0.3071 - val_loss: 2.6710 - learning_rate: 1.0000e-04
Epoch 3/12
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 325ms/step - accuracy: 0.2294 - loss: 2.7558
Epoch 3: val_loss improved from 2.67096 to 2.46148, saving model to best_plantdoc_modeltrue.keras

Epoch 3: finished saving mod

In [ ]:
# Unfreeszing layers to increase accuracy of model
base_model.trainable = True

fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:

history_fine = model.fit(
    train_data,
    validation_data=valid_data,
    epochs=15,
    callbacks=[checkpoint, lr_reduction]
)

Epoch 1/15
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 308ms/step - accuracy: 0.2448 - loss: 2.6450
Epoch 1: val_loss improved from 1.71764 to 1.67438, saving model to best_plantdoc_modeltrue.keras

Epoch 1: finished saving model to best_plantdoc_modeltrue.keras
67/67 ━━━━━━━━━━━━━━━━━━━━ 42s 421ms/step - accuracy: 0.2669 - loss: 2.5882 - val_accuracy: 0.5056 - val_loss: 1.6744 - learning_rate: 1.0000e-05
Epoch 2/15
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 331ms/step - accuracy: 0.3184 - loss: 2.3792
Epoch 2: val_loss improved from 1.67438 to 1.65302, saving model to best_plantdoc_modeltrue.keras

Epoch 2: finished saving model to best_plantdoc_modeltrue.keras
67/67 ━━━━━━━━━━━━━━━━━━━━ 27s 407ms/step - accuracy: 0.3418 - loss: 2.3181 - val_accuracy: 0.5243 - val_loss: 1.6530 - learning_rate: 1.0000e-05
Epoch 3/15
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 331ms/step - accuracy: 0.3717 - loss: 2.2088
Epoch 3: val_loss improved from 1.65302 to 1.63983, saving model to best_plantdoc_modeltrue.keras

Epoch 3: finished saving 

In [ ]:

base_model.trainable = True

fine_tune_at = 50
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:

history_deep_fine = model.fit(
    train_data,
    validation_data=valid_data,
    epochs=25,
    callbacks=[checkpoint, lr_reduction]
)


Epoch 1/25
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 325ms/step - accuracy: 0.5317 - loss: 1.5264
Epoch 1: val_loss improved from 1.47464 to 1.46470, saving model to best_plantdoc_modeltrue.keras

Epoch 1: finished saving model to best_plantdoc_modeltrue.keras
67/67 ━━━━━━━━━━━━━━━━━━━━ 43s 440ms/step - accuracy: 0.5290 - loss: 1.5366 - val_accuracy: 0.5581 - val_loss: 1.4647 - learning_rate: 1.0000e-05
Epoch 2/25
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 338ms/step - accuracy: 0.5388 - loss: 1.5235
Epoch 2: val_loss improved from 1.46470 to 1.44930, saving model to best_plantdoc_modeltrue.keras

Epoch 2: finished saving model to best_plantdoc_modeltrue.keras
67/67 ━━━━━━━━━━━━━━━━━━━━ 39s 424ms/step - accuracy: 0.5262 - loss: 1.5425 - val_accuracy: 0.5581 - val_loss: 1.4493 - learning_rate: 1.0000e-05
Epoch 3/25
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 332ms/step - accuracy: 0.5237 - loss: 1.5211
Epoch 3: val_loss improved from 1.44930 to 1.43316, saving model to best_plantdoc_modeltrue.keras

Epoch 3: finished saving 

In [ ]:
# Extended fine-tuning to chase 65%+
history_extended = model.fit(
    train_data,
    validation_data=valid_data,
    epochs=40,
    callbacks=[checkpoint, lr_reduction]
)

Epoch 1/40
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 335ms/step - accuracy: 0.6760 - loss: 1.0305
Epoch 1: val_loss improved from 1.24239 to 1.23862, saving model to best_plantdoc_modeltrue.keras

Epoch 1: finished saving model to best_plantdoc_modeltrue.keras
67/67 ━━━━━━━━━━━━━━━━━━━━ 28s 411ms/step - accuracy: 0.6606 - loss: 1.0814 - val_accuracy: 0.6086 - val_loss: 1.2386 - learning_rate: 1.0000e-05
Epoch 2/40
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 327ms/step - accuracy: 0.6513 - loss: 1.1007
Epoch 2: val_loss improved from 1.23862 to 1.23591, saving model to best_plantdoc_modeltrue.keras

Epoch 2: finished saving model to best_plantdoc_modeltrue.keras
67/67 ━━━━━━━━━━━━━━━━━━━━ 28s 418ms/step - accuracy: 0.6648 - loss: 1.0830 - val_accuracy: 0.6086 - val_loss: 1.2359 - learning_rate: 1.0000e-05
Epoch 3/40
67/67 ━━━━━━━━━━━━━━━━━━━━ 0s 325ms/step - accuracy: 0.6848 - loss: 0.9909
Epoch 3: val_loss improved from 1.23591 to 1.23497, saving model to best_plantdoc_modeltrue.keras

Epoch 3: finished saving 

In [ ]:
from google.colab import files

files.download('best_plantdoc_modeltrue.keras')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print(train_data.class_names)


['Apple_Scab_Leaf', 'Apple_leaf', 'Apple_rust_leaf', 'Bell_pepper_leaf', 'Bell_pepper_leaf_spot', 'Blueberry_leaf', 'Cherry_leaf', 'Corn_Gray_leaf_spot', 'Corn_leaf_blight', 'Corn_rust_leaf', 'Peach_leaf', 'Potato_leaf_early_blight', 'Potato_leaf_late_blight', 'Raspberry_leaf', 'Soyabean_leaf', 'Squash_Powdery_mildew_leaf', 'Strawberry_leaf', 'Tomato_Early_blight_leaf', 'Tomato_Septoria_leaf_spot', 'Tomato_leaf', 'Tomato_leaf_bacterial_spot', 'Tomato_leaf_late_blight', 'Tomato_leaf_mosaic_virus', 'Tomato_leaf_yellow_virus', 'Tomato_mold_leaf', 'Tomato_two_spotted_spider_mites_leaf', 'grape_leaf', 'grape_leaf_black_rot']
